# Inventarios 360

## Inicio del proyecto BI

Este notebook inicia el trabajo analítico sobre el archivo `Fechas de Vencimiento Beauty Care 1.xlsx`.

En esta primera versión se implementan dos reglas base del proyecto:

- identificador compuesto del producto por `Item ID + Contenedor`;
- cálculo de `dias_en_inventario` usando una `fecha_corte` explícita menos `Fecha de Ingreso`.


## 1. Imports y configuración

Si `openpyxl` no está instalado en el entorno, ejecutar primero la instalación.


In [1]:
# Inventarios 360 — Configuración del entorno
import sys
from pathlib import Path

# Asegurar que src/ esté en el path (ejecutar kernel desde la raíz del repo)
repo_root = Path.cwd()
assert (repo_root / "src").is_dir(), (
    f"src/ no encontrado bajo {repo_root} — ejecutar el kernel desde la raíz del proyecto"
)
sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv()

from src.etl import FECHA_CORTE, UMBRAL_CRITICO, UMBRAL_PROXIMO
print(f"fecha_corte    : {FECHA_CORTE}")
print(f"umbral_critico : {UMBRAL_CRITICO} días")
print(f"umbral_proximo : {UMBRAL_PROXIMO} días")

fecha_corte    : 2025-04-01
umbral_critico : 15 días
umbral_proximo : 30 días


In [2]:
# Ejecución del pipeline ETL
from src.etl.pipeline import run_pipeline

# load_to_db=True requiere variables de entorno configuradas en .env
df_clean, validation_report = run_pipeline(load_to_db=False)
df_clean.head()

=== ETL Inventarios 360 ===
  Fuente      : datos_proyecto/Fechas de Vencimiento Beauty Care 1.xlsx
  Hoja        : Page1_1
  fecha_corte : 2025-04-01

[1/4] Extrayendo datos...
      15,950 registros leídos, 13 columnas.
[2/4] Transformando...
      15,950 registros limpios | 0 descartados.
[3/4] Validando...
Validacion completada: 3 advertencia(s) encontradas.
  [duplicados] product_container_id: 197 duplicados encontrados. Revisar si la granularidad del archivo tiene múltiples contenedores por ítem.
  [fechas] 15751 registros con fecha_ingreso futura (> 2025-04-01). Se preservan con calidad_flag='fecha_ingreso_futura'.
  [fechas] 37 registros con fecha_vencimiento nula.

  Distribución de estados:
    vigente                 15,913  (99.8%)
    sin_fecha                   37  (0.2%)

[4/4] Carga a DB omitida (load_to_db=False).

=== Pipeline completado ===


,marca,item_id,id_inventario,descripcion,categoria,unds,fecha_ingreso,fecha_vencimiento,dias_antes_vencimiento_raw,contenedor,...,row_null_pct,product_container_id,dias_en_inventario,dias_para_vencimiento,estado_inventario,segmento_rotacion,score_riesgo,mes_vencimiento,anio_vencimiento,calidad_flag
0,ZOII,707215,NaN,TRATAMIENTO RIZOS,CAPILAR,50.0,2026-03-14,2028-02-15,678,L05-10A08,...,0.0,707215_L05-10A08,-347,1050.0,vigente,sin_dato,0,2.0,2028.0,fecha_ingreso_futura
1,ZOII,707215,NaN,TRATAMIENTO RIZOS,CAPILAR,59.0,2026-03-13,2028-02-15,678,L01-05A08,...,0.0,707215_L01-05A08,-346,1050.0,vigente,sin_dato,0,2.0,2028.0,fecha_ingreso_futura
2,ZOII,707215,NaN,TRATAMIENTO RIZOS,CAPILAR,1.0,2026-03-20,2028-02-15,678,LPN0007408601,...,0.0,707215_LPN0007408601,-353,1050.0,vigente,sin_dato,0,2.0,2028.0,fecha_ingreso_futura
3,ZOII,707215,NaN,TRATAMIENTO RIZOS,CAPILAR,30.0,2026-03-05,2028-03-04,696,LPN0007311358,...,0.0,707215_LPN0007311358,-338,1068.0,vigente,sin_dato,0,3.0,2028.0,fecha_ingreso_futura
4,ZOII,707215,NaN,TRATAMIENTO RIZOS,CAPILAR,30.0,2026-03-05,2028-03-04,696,LPN0007310332,...,0.0,707215_LPN0007310332,-338,1068.0,vigente,sin_dato,0,3.0,2028.0,fecha_ingreso_futura


In [3]:
print(f"Shape: {df_clean.shape}")
print(f"\nTipos:")
print(df_clean.dtypes)
print(f"\nNulos por columna:")
print(df_clean.isnull().sum())
print(f"\nDistribución de calidad_flag:")
print(df_clean["calidad_flag"].value_counts())
print(f"\nDistribución de estado_inventario:")
print(df_clean["estado_inventario"].value_counts())
print(f"\nRango de score_riesgo: {df_clean['score_riesgo'].min()} – {df_clean['score_riesgo'].max()}")

Shape: (15950, 22)

Tipos:
marca                                 object
item_id                               object
id_inventario                         object
descripcion                           object
categoria                             object
unds                                 float64
fecha_ingreso                 datetime64[ns]
fecha_vencimiento             datetime64[ns]
dias_antes_vencimiento_raw            object
contenedor                            object
rotacion_raw                          object
estado_raw                            object
row_null_pct                         float64
product_container_id                  object
dias_en_inventario                     int64
dias_para_vencimiento                float64
estado_inventario                     object
segmento_rotacion                     object
score_riesgo                           int64
mes_vencimiento                      float64
anio_vencimiento                     float64
calidad_flag                

In [5]:
# %pip install openpyxl

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

fecha_corte = pd.Timestamp("2026-04-10")
data_path = Path("datos_proyecto") / "Fechas de Vencimiento Beauty Care 1.xlsx"
sheet_name = "Page1_1"

print(f"Archivo: {data_path}")
print(f"Hoja: {sheet_name}")
print(f"Fecha de corte: {fecha_corte.date()}")


Archivo: datos_proyecto/Fechas de Vencimiento Beauty Care 1.xlsx
Hoja: Page1_1
Fecha de corte: 2026-04-10


## 2. Carga de datos

Se carga la hoja principal del inventario y se hace una normalización inicial de columnas.


In [6]:
df_raw = pd.read_excel(data_path, sheet_name=sheet_name)
df_raw.columns = df_raw.columns.str.strip()

print(df_raw.shape)
df_raw.head()


(15950, 12)


,Marca,Item ID,ID Inventario,Descripción,Categoría,Unds,Fecha de Ingreso,Fecha Vencimiento,Dias antes de Vencimiento,Contenedor,Rotación,Estado
0,ZOII,707215,NaN,TRATAMIENTO RIZOS,CAPILAR,50,2026-03-14,2028-02-15,678,L05-10A08,NaN,NaN
1,ZOII,707215,NaN,TRATAMIENTO RIZOS,CAPILAR,59,2026-03-13,2028-02-15,678,L01-05A08,NaN,NaN
2,ZOII,707215,NaN,TRATAMIENTO RIZOS,CAPILAR,1,2026-03-20,2028-02-15,678,LPN0007408601,NaN,NaN
3,ZOII,707215,NaN,TRATAMIENTO RIZOS,CAPILAR,30,2026-03-05,2028-03-04,696,LPN0007311358,NaN,NaN
4,ZOII,707215,NaN,TRATAMIENTO RIZOS,CAPILAR,30,2026-03-05,2028-03-04,696,LPN0007310332,NaN,NaN


## 3. Estandarización mínima

En esta etapa se convierten las columnas críticas a tipos utilizables para análisis.


In [7]:
df = df_raw.copy()

df["Item ID"] = df["Item ID"].astype("string").str.strip()
df["Contenedor"] = df["Contenedor"].astype("string").str.strip()
df["Unds"] = pd.to_numeric(df["Unds"], errors="coerce")
df["Fecha de Ingreso"] = pd.to_datetime(df["Fecha de Ingreso"], errors="coerce")
df["Fecha Vencimiento"] = pd.to_datetime(df["Fecha Vencimiento"], errors="coerce")

df[["Item ID", "Contenedor", "Unds", "Fecha de Ingreso", "Fecha Vencimiento"]].head()


,Item ID,Contenedor,Unds,Fecha de Ingreso,Fecha Vencimiento
0,707215,L05-10A08,50,2026-03-14,2028-02-15
1,707215,L01-05A08,59,2026-03-13,2028-02-15
2,707215,LPN0007408601,1,2026-03-20,2028-02-15
3,707215,LPN0007311358,30,2026-03-05,2028-03-04
4,707215,LPN0007310332,30,2026-03-05,2028-03-04


## 4. Identificador compuesto

El identificador operativo del registro se construye con `Item ID + Contenedor`.


In [8]:
df["product_container_id"] = (
    df["Item ID"].fillna("SIN_ITEM").astype("string").str.strip()
    + "_"
    + df["Contenedor"].fillna("SIN_CONTENEDOR").astype("string").str.strip()
)

df[["Item ID", "Contenedor", "product_container_id"]].head()


,Item ID,Contenedor,product_container_id
0,707215,L05-10A08,707215_L05-10A08
1,707215,L01-05A08,707215_L01-05A08
2,707215,LPN0007408601,707215_LPN0007408601
3,707215,LPN0007311358,707215_LPN0007311358
4,707215,LPN0007310332,707215_LPN0007310332


## 5. Antigüedad en inventario

Se calcula cuántos días lleva el inventario almacenado usando la fecha de corte definida para el proyecto.


In [9]:
df["dias_en_inventario"] = (fecha_corte - df["Fecha de Ingreso"]).dt.days
df["dias_para_vencimiento"] = (df["Fecha Vencimiento"] - fecha_corte).dt.days

df[[
    "product_container_id",
    "Fecha de Ingreso",
    "Fecha Vencimiento",
    "dias_en_inventario",
    "dias_para_vencimiento",
]].head()


,product_container_id,Fecha de Ingreso,Fecha Vencimiento,dias_en_inventario,dias_para_vencimiento
0,707215_L05-10A08,2026-03-14,2028-02-15,27,676.0
1,707215_L01-05A08,2026-03-13,2028-02-15,28,676.0
2,707215_LPN0007408601,2026-03-20,2028-02-15,21,676.0
3,707215_LPN0007311358,2026-03-05,2028-03-04,36,694.0
4,707215_LPN0007310332,2026-03-05,2028-03-04,36,694.0


## 6. Validación rápida

Se revisa el resultado de las variables nuevas para confirmar que ya existe una base mínima para análisis de riesgo.


In [10]:
df[["dias_en_inventario", "dias_para_vencimiento", "Unds"]].describe(include="all")


,dias_en_inventario,dias_para_vencimiento,Unds
count,15950.000000,15913.000000,15950.000000
mean,60.835862,615.851317,36.937179
std,69.770537,234.997552,127.212621
min,20.000000,-98.000000,1.000000
25%,23.000000,504.000000,10.000000
50%,36.000000,685.000000,20.000000
75%,53.000000,705.000000,30.000000
max,708.000000,2156.000000,11515.000000


In [11]:
df[[
    "Marca",
    "Item ID",
    "Descripción",
    "Contenedor",
    "product_container_id",
    "dias_en_inventario",
    "dias_para_vencimiento",
    "Unds",
]].head(10)


,Marca,Item ID,Descripción,Contenedor,product_container_id,dias_en_inventario,dias_para_vencimiento,Unds
0,ZOII,707215,TRATAMIENTO RIZOS,L05-10A08,707215_L05-10A08,27,676.0,50
1,ZOII,707215,TRATAMIENTO RIZOS,L01-05A08,707215_L01-05A08,28,676.0,59
2,ZOII,707215,TRATAMIENTO RIZOS,LPN0007408601,707215_LPN0007408601,21,676.0,1
3,ZOII,707215,TRATAMIENTO RIZOS,LPN0007311358,707215_LPN0007311358,36,694.0,30
4,ZOII,707215,TRATAMIENTO RIZOS,LPN0007310332,707215_LPN0007310332,36,694.0,30
5,ZOII,707215,TRATAMIENTO RIZOS,LPN0007311965,707215_LPN0007311965,36,694.0,30
6,ZOII,707215,TRATAMIENTO RIZOS,LPN0007310349,707215_LPN0007310349,36,694.0,30
7,ZOII,707215,TRATAMIENTO RIZOS,LPN0007311357,707215_LPN0007311357,36,694.0,30
8,ZOII,707215,TRATAMIENTO RIZOS,LPN0007311992,707215_LPN0007311992,36,694.0,30
9,ZOII,707215,TRATAMIENTO RIZOS,LPN0007310338,707215_LPN0007310338,36,694.0,30
